# Mendeley audio and transcript review

Run `scripts/mendeley_pipeline.py` first. This notebook reads only the local Mendeley output and shows the source filename beside every transcript.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

root = next(path for path in (Path("data"), Path("../data")) if (path / "karakalpak-mendeley" / "aligned" / "manifest.csv").is_file())
base = root / "karakalpak-mendeley"
corpus = base / "aligned"
cleaned_dir = base / "cleaned"
manifest = pd.read_csv(corpus / "manifest.csv")
measurements = pd.read_csv(corpus / "measurements.csv")
cleaned = pd.read_csv(cleaned_dir / "metadata.csv")
review = pd.read_csv(corpus / "abbreviation_review.csv").fillna("")


In [ ]:
print(f"Source pairs: {len(manifest):,}; measured pairs: {len(measurements):,}; cleaned pairs: {len(cleaned):,}")
for label, folder in (
    ("aligned original", corpus / "audio_original"),
    ("aligned 16 kHz", corpus / "audio_16k"),
    ("cleaned training audio", cleaned_dir / "audio"),
    ("cleaned original audio", cleaned_dir / "audio_original"),
):
    print(label, len(list(folder.glob("*.wav"))))


## Play a matched pair

Change `selected_index` to inspect any row. The text and both playable files use the same source filename.


In [ ]:
selected_index = 0
selected = measurements.iloc[selected_index]
print(selected[["audio_index", "filename", "sentence_latin", "duration", "snr_original", "snr_16k"]].to_string())
display(Audio(filename=str(corpus / "audio_original" / selected["filename"])))
display(Audio(filename=str(corpus / "audio_16k" / selected["filename"])))


## Original or decoded WAV


In [ ]:
before, before_sr = librosa.load(corpus / "audio_original" / selected["filename"], sr=None)
before_frame = round(0.025 * before_sr)
before_hop = round(0.010 * before_sr)
before_intervals = librosa.effects.split(before, top_db=30, frame_length=before_frame, hop_length=before_hop)
plt.figure(figsize=(12, 2.5))
plt.plot(np.arange(len(before)) / before_sr, before, lw=0.5)
for start, end in before_intervals:
    plt.axvspan(start / before_sr, end / before_sr, color="tab:green", alpha=0.12)
plt.title(f"Original waveform and speech regions — {before_sr:,} Hz")
plt.xlabel("Seconds")
plt.show()


In [ ]:
before_stft = np.abs(librosa.stft(before, n_fft=before_frame, hop_length=before_hop))
plt.figure(figsize=(12, 3))
librosa.display.specshow(librosa.amplitude_to_db(before_stft, ref=np.max), sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="linear")
plt.title("Original STFT (dB)")
plt.colorbar()
plt.show()


In [ ]:
before_mel = librosa.feature.melspectrogram(y=before, sr=before_sr, n_fft=before_frame, hop_length=before_hop, n_mels=80)
plt.figure(figsize=(12, 3))
librosa.display.specshow(librosa.power_to_db(before_mel, ref=np.max), sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="mel", vmin=-80, vmax=0)
plt.title("Original Mel spectrogram (dB)")
plt.colorbar()
plt.show()


In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.specshow(np.log(before_mel + 1e-10), sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="mel")
plt.title("Original log Mel filterbank")
plt.colorbar()
plt.show()


## 16 kHz mono WAV


In [ ]:
after, after_sr = librosa.load(corpus / "audio_16k" / selected["filename"], sr=None)
after_frame = round(0.025 * after_sr)
after_hop = round(0.010 * after_sr)
after_intervals = librosa.effects.split(after, top_db=30, frame_length=after_frame, hop_length=after_hop)
plt.figure(figsize=(12, 2.5))
plt.plot(np.arange(len(after)) / after_sr, after, lw=0.5)
for start, end in after_intervals:
    plt.axvspan(start / after_sr, end / after_sr, color="tab:green", alpha=0.12)
plt.title("16 kHz waveform and speech regions")
plt.xlabel("Seconds")
plt.show()


In [ ]:
after_stft = np.abs(librosa.stft(after, n_fft=after_frame, hop_length=after_hop))
plt.figure(figsize=(12, 3))
librosa.display.specshow(librosa.amplitude_to_db(after_stft, ref=np.max), sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="linear")
plt.title("16 kHz STFT (dB)")
plt.colorbar()
plt.show()


In [ ]:
after_mel = librosa.feature.melspectrogram(y=after, sr=after_sr, n_fft=after_frame, hop_length=after_hop, n_mels=80)
plt.figure(figsize=(12, 3))
librosa.display.specshow(librosa.power_to_db(after_mel, ref=np.max), sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="mel", vmin=-80, vmax=0)
plt.title("16 kHz Mel spectrogram (dB)")
plt.colorbar()
plt.show()


In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.specshow(np.log(after_mel + 1e-10), sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="mel")
plt.title("16 kHz log Mel filterbank")
plt.colorbar()
plt.show()


## Cleaning and abbreviation review

`cleaned/train.csv` pairs each normalized transcript with one file in `cleaned/audio/`. `cleaned/metadata.csv` retains source traceability. Play any review row to inspect a spoken abbreviation.


In [ ]:
print("Minimum duration:", measurements["duration"].min())
print("Minimum source word count:", measurements["sentence_latin"].str.split().str.len().min())
print("Measured SNR values:", measurements[["snr_original", "snr_16k"]].notna().sum().to_dict())
print("Applied abbreviation expansions:", (review["verified"] == "yes").sum())
display(review)


In [ ]:
review_index = 0
candidate = review.iloc[review_index]
source_row = manifest.loc[manifest["filename"] == candidate["source_filename"]].iloc[0]
cleaned_row = cleaned.loc[cleaned["source_filename"] == candidate["source_filename"]].iloc[0]
print("Source:", source_row["sentence_latin"])
print("Cleaned:", cleaned_row["sentence_latin"])
print("Decision:", candidate["verified"], candidate["evidence"])
display(Audio(filename=str(corpus / "audio_16k" / candidate["source_filename"])))
